# Compile finalized per-block saccades

Load saccade CSVs written by the preprocessing GUI **Saccades** tab
(`block/analysis/saccades/`) across a paper registry, QC them, and optionally
hand off to `PaperContext` / write a combined pickle for the flexible paper tool.

**Prerequisites:** each block finalized via the Saccades tab (or equivalent
`write_finalized_saccades`). Blocks without `analysis/saccades/` are reported
and skipped.

See also: `flexible_paper_figures_tool.ipynb` (`USE_FINALIZED_SACCADES=True`).

## 0. Setup

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO = Path.cwd()
if not (REPO / "src" / "eye_tracking_system_tools").is_dir():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "eye_tracking_system_tools").is_dir():
            REPO = p
            break

sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

from eye_tracking_system_tools.analysis.block_registry import load_registry
from eye_tracking_system_tools.analysis.event_cache import save_event_cache
from eye_tracking_system_tools.analysis.export_meta import load_params_yaml
from eye_tracking_system_tools.analysis.paper_gui import PaperContext, block_qc_table
from eye_tracking_system_tools.analysis.pipeline import build_event_tables
from eye_tracking_system_tools.analysis.pixel_calibration import has_pixel_calibration
from eye_tracking_system_tools.analysis.run_layout import resolve_run_dir
from eye_tracking_system_tools.analysis.saccade_export import (
    detection_params_fingerprint,
    has_finalized_saccades,
    read_finalized_saccades,
    saccades_dir,
)

print("REPO:", REPO)

REPO: /Users/nimi/Projects/PETS


## 1. Paths & registry

In [2]:
# --- edit me ---
# REGISTRY = REPO / "configs" / "paper_blocks.yaml"
REGISTRY = REPO / "configs" / "paper_blocks_custom.yaml"
# REGISTRY = REPO / "configs" / "sample_blocks.yaml"
PARAMS = REPO / "configs" / "analysis_params.yaml"
TAG = "compiled_saccades"
# ---------------

params = load_params_yaml(PARAMS)
specs = load_registry(REGISTRY)
run = resolve_run_dir(REPO / "outputs", tag=TAG, prefix="paper")
print("REGISTRY:", REGISTRY)
print("blocks:", len(specs))
print("run_dir:", run.run_dir)

REGISTRY: /Users/nimi/Projects/PETS/configs/paper_blocks_custom.yaml
blocks: 4
run_dir: /Users/nimi/Projects/PETS/outputs/paper_compiled_saccades


## 2. Inventory: which blocks have finalized saccades?

In [3]:
rows = []
for spec in specs:
    has = has_finalized_saccades(spec.block_path)
    row = {
        "block_key": spec.block_key,
        "has_finalized": has,
        "has_pix_size": has_pixel_calibration(spec.block_path),
        "saccades_dir": str(saccades_dir(spec.block_path)),
    }
    if has:
        fin = read_finalized_saccades(spec.block_path)
        row["n_events"] = len(fin.all_saccades)
        row["n_synced"] = len(fin.synced)
        row["n_monocular"] = len(fin.non_synced)
        row["params_fp"] = detection_params_fingerprint(fin.params)
        row["source"] = fin.source
        row["isi_mean_ms"] = fin.summary.get("isi_mean_ms")
    rows.append(row)

inventory = pd.DataFrame(rows)
display(inventory)
missing = inventory.loc[~inventory["has_finalized"], "block_key"].tolist()
print(f"finalized: {inventory['has_finalized'].sum()} / {len(inventory)}")
if missing:
    print("missing:", ", ".join(missing))

,block_key,has_finalized,has_pix_size,saccades_dir,n_events,n_synced,n_monocular,params_fp,source,isi_mean_ms
0,M_002_block_012,True,True,/Volumes/Data/Nimrod/experiments/M_002/2026_07...,9068,6090,2978,8e3cdd879e9d,pickle,224.277325
1,M_002_block_013,True,True,/Volumes/Data/Nimrod/experiments/M_002/2026_07...,4067,2480,1587,dd4b8080a689,pickle,551.028981
2,M_002_block_014,True,True,/Volumes/Data/Nimrod/experiments/M_002/2026_07...,8853,4852,4001,f1cf78c51cca,pickle,215.182983
3,M_002_block_015,True,True,/Volumes/Data/Nimrod/experiments/M_002/2026_07...,2860,1520,1340,8e83fb96bd36,pickle,273.626370


finalized: 4 / 4


## 3. Build EventTables (prefer finalized CSVs)

`build_event_tables(..., prefer_finalized=True)` loads `analysis/saccades/` when
present and falls back to on-the-fly detection otherwise.

In [4]:
tables = build_event_tables(
    specs,
    params=params,
    keep_traces=False,
    prefer_finalized=True,
)
print(
    f"blocks={len(tables.blocks)}  all_saccades={len(tables.all_saccades)}  "
    f"synced_rows={len(tables.synced)}  non_synced={len(tables.non_synced)}"
)

qc = block_qc_table(tables)
display(qc)

cache_path = save_event_cache(tables, run.metadata_dir, specs=specs)
print("cached →", cache_path)

ctx = PaperContext(
    tables,
    run.run_dir,
    registry_path=REGISTRY,
    params_path=PARAMS,
)
print("PaperContext ready — open flexible_paper_figures_tool.ipynb or build figures here.")

[M_002_block_012] WARNING: finalized saccade params differ from run YAML — using on-disk events anyway
[M_002_block_012] finalized saccades n=9068 synced_pairs≈3045 non_synced=2978 (from pickle)
[M_002_block_013] WARNING: finalized saccade params differ from run YAML — using on-disk events anyway
[M_002_block_013] finalized saccades n=4067 synced_pairs≈1240 non_synced=1587 (from pickle)
[M_002_block_014] WARNING: finalized saccade params differ from run YAML — using on-disk events anyway
[M_002_block_014] finalized saccades n=8853 synced_pairs≈2426 non_synced=4001 (from pickle)
[M_002_block_015] WARNING: finalized saccade params differ from run YAML — using on-disk events anyway
[M_002_block_015] finalized saccades n=2860 synced_pairs≈760 non_synced=1340 (from pickle)
blocks=4  all_saccades=24848  synced_rows=14942  non_synced=9906


,block_key,animal,block,n_saccades_L,n_saccades_R,n_synced_pairs,has_traces,has_behavior_state,has_pix_size,block_path
0,M_002_block_012,M_002,block_012,4499,4569,3045,False,False,True,/Volumes/Data/Nimrod/experiments/M_002/2026_07...
1,M_002_block_013,M_002,block_013,2026,2041,1240,False,False,True,/Volumes/Data/Nimrod/experiments/M_002/2026_07...
2,M_002_block_014,M_002,block_014,4393,4460,2426,False,False,True,/Volumes/Data/Nimrod/experiments/M_002/2026_07...
3,M_002_block_015,M_002,block_015,1216,1644,760,False,False,True,/Volumes/Data/Nimrod/experiments/M_002/2026_07...


cached → /Users/nimi/Projects/PETS/outputs/paper_compiled_saccades/metadata/event_cache/5ba2f66b39b168326c3a578fdbb44be642564401.pkl
PaperContext ready — open flexible_paper_figures_tool.ipynb or build figures here.
